# SOV Extraction with Azure AI Content Understanding

Extract Statement of Values (SOV) data from the 6 broker submission attachments under `demo/sov/attachments/` (4 `.xlsx` + 2 `.pdf`) using a single **custom analyzer** built from an analyzer template JSON.

**Approach**

- Uses the GA `azure-ai-contentunderstanding` Python SDK.
- One analyzer template at [reference/analyzer-templates/sov_extraction.json](../reference/analyzer-templates/sov_extraction.json), all fields use `method: "extract"` (verbatim extraction with grounding + confidence).
- **Replay mode by default**: if a cached result exists under `reference/cu-output/` the API call is skipped. Delete the cached file (or set `FORCE_REFRESH = True`) to re-run extraction.
- Same analyzer runs over all 6 templates regardless of layout (header-block, flat table, multi-sheet, native PDF, scanned PDF, messy broker spreadsheet).

**Prerequisites**

1. A Microsoft Foundry resource with Content Understanding enabled.
2. `gpt-4.1` and `text-embedding-3-large` model deployments configured as defaults (see [`sample_update_defaults.py`](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_update_defaults.py)).
3. `demo/sov/.env` populated from `.env.example` with `CONTENTUNDERSTANDING_ENDPOINT` and either `CONTENTUNDERSTANDING_KEY` or `az login` for `DefaultAzureCredential`.


## 1. Setup and imports

Load env vars, import the SDK, define paths.


In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import ContentAnalyzer
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

# --- Paths ---
NB_DIR        = Path.cwd() if "__file__" not in globals() else Path(__file__).parent
DEMO_ROOT     = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
ATTACH_DIR    = DEMO_ROOT / "attachments"
TEMPLATE_PATH = DEMO_ROOT / "reference" / "analyzer-templates" / "sov_extraction.json"
OUTPUT_DIR    = DEMO_ROOT / "reference" / "cu-output"
ENV_PATH      = DEMO_ROOT / ".env"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Env ---
load_dotenv(ENV_PATH)

# --- Behavior switches ---
FORCE_REFRESH = True  # set True to re-run all extractions even if cached results exist

print(f"Demo root:     {DEMO_ROOT}")
print(f"Attachments:   {ATTACH_DIR}  ({len(list(ATTACH_DIR.glob('*')))} files)")
print(f"Template:      {TEMPLATE_PATH}")
print(f"Output cache:  {OUTPUT_DIR}")
print(f"FORCE_REFRESH: {FORCE_REFRESH}")


Demo root:     c:\Users\jomedin\Documents\azure-ai-foundry-insurance\demo\sov
Attachments:   c:\Users\jomedin\Documents\azure-ai-foundry-insurance\demo\sov\attachments  (6 files)
Template:      c:\Users\jomedin\Documents\azure-ai-foundry-insurance\demo\sov\reference\analyzer-templates\sov_extraction.json
Output cache:  c:\Users\jomedin\Documents\azure-ai-foundry-insurance\demo\sov\reference\cu-output
FORCE_REFRESH: True


## 2. Build the Content Understanding client

Authenticate with the Foundry resource. Uses `CONTENTUNDERSTANDING_KEY` if set in the env file, otherwise falls back to `DefaultAzureCredential` (run `az login` first). The client is instantiated lazily so the rest of the notebook can run from cached results without an Azure connection.


In [2]:
_client: ContentUnderstandingClient | None = None


def get_client() -> ContentUnderstandingClient:
    """Lazily build a ContentUnderstandingClient. Authenticates with DefaultAzureCredential
    (uses your `az login` session, VS Code account, or managed identity)."""
    global _client
    if _client is not None:
        return _client

    endpoint = (
        os.environ.get("CONTENTUNDERSTANDING_ENDPOINT")
        or os.environ.get("FOUNDRY_ENDPOINT")
    )
    if not endpoint:
        raise RuntimeError(
            "Set FOUNDRY_ENDPOINT (or CONTENTUNDERSTANDING_ENDPOINT) in demo/sov/.env."
        )

    cred = DefaultAzureCredential()
    _client = ContentUnderstandingClient(endpoint=endpoint, credential=cred)
    print(f"✅ Client ready  | endpoint={endpoint}  | auth=DefaultAzureCredential")
    return _client


## 3. Load the analyzer template

The analyzer template ([sov_extraction.json](../reference/analyzer-templates/sov_extraction.json)) is a plain JSON file in the same shape used by the [Azure Samples analyzer_templates](https://github.com/Azure-Samples/azure-ai-content-understanding-python/tree/main/analyzer_templates) folder. Field-level descriptions act as prompts for the model to handle label drift across templates (`Bldg Val` / `Building Value` / `RC Bldg`).


In [3]:
# Two analyzer templates, three input shapes:
#   - sov_extraction.json          -> PDF, method=extract  (grounding + confidence)
#   - sov_extraction_generate.json -> Excel + per-image fan-outs, method=generate
#
# Approach 3 (xlsx with embedded images) reuses the same generate analyzer for
# the workbook AND each embedded image, then merges Locations[] client-side.
# (Pro Mode would do this in a single call but is preview-only and not
# available in our resource's region; see notebooks/README.md.)
TEMPLATE_DIR = DEMO_ROOT / "reference" / "analyzer-templates"

with open(TEMPLATE_DIR / "sov_extraction.json", "r", encoding="utf-8") as f:
    template_extract = json.load(f)
with open(TEMPLATE_DIR / "sov_extraction_generate.json", "r", encoding="utf-8") as f:
    template_generate = json.load(f)

# Generic analyzer IDs (no customer name). Bump V<N> whenever the field
# descriptions in the template JSON change so a fresh analyzer is created.
ANALYZER_EXTRACT_ID  = "sovExtractV1"
ANALYZER_GENERATE_ID = "sovGenerateV1"

print(f"{ANALYZER_EXTRACT_ID:24s}  base={template_extract['baseAnalyzerId']:30s} (PDF path)")
print(f"{ANALYZER_GENERATE_ID:24s}  base={template_generate['baseAnalyzerId']:30s} (Excel + image fan-out)")


sovExtractV1              base=prebuilt-document              (PDF path)
sovGenerateV1             base=prebuilt-document              (Excel + image fan-out)


## 4. Create the analyzer

Creates the custom analyzer in the Foundry resource with `begin_create_analyzer`. We pass the loaded template dict directly as the resource — no model dataclasses required. Idempotent: if an analyzer with the same id already exists we reuse it.


In [4]:
def ensure_analyzer(analyzer_id: str, template: dict) -> str:
    """Create the analyzer if it doesn't already exist. Returns the analyzer id."""
    client = get_client()
    try:
        existing = client.get_analyzer(analyzer_id=analyzer_id)
        print(f"♻️  Reusing analyzer '{analyzer_id}'  (status: {existing.status})")
        return analyzer_id
    except Exception:
        pass

    print(f"🛠️  Creating analyzer '{analyzer_id}'...")
    poller = client.begin_create_analyzer(analyzer_id=analyzer_id, resource=template)
    poller.result()
    print(f"✅ Analyzer '{analyzer_id}' created.")
    return analyzer_id


# Uncomment to create both analyzers up front. Otherwise they'll be created
# lazily on first cache miss in section 6.
# for aid, tpl in ANALYZER_BY_SUFFIX.values():
#     ensure_analyzer(aid, tpl)


## 5. Discover the SOV attachments

We extract from any `.xlsx` or `.pdf` under `attachments/` so this notebook stays generic — drop more SOVs in there and the loop below will pick them up.


In [5]:
SUPPORTED_SUFFIXES = {".xlsx", ".pdf"}

attachments = sorted(
    p for p in ATTACH_DIR.iterdir() if p.suffix.lower() in SUPPORTED_SUFFIXES
)

pd.DataFrame(
    [
        {
            "file": p.name,
            "type": p.suffix.lower().lstrip("."),
            "size_kb": round(p.stat().st_size / 1024, 1),
            "cached": (OUTPUT_DIR / f"{p.stem}.json").exists(),
        }
        for p in attachments
    ]
)


,file,type,size_kb,cached
0,01_acme_SOV.xlsx,xlsx,23.6,True
1,02_cascade_SOV.xlsx,xlsx,6.0,True
2,03_magnolia_SOV.xlsx,xlsx,10.4,True
3,04_summit_SOV.pdf,pdf,6.1,True
4,05_heartland_SOV.pdf,pdf,189.5,True
5,06_coastal_SOV.xlsx,xlsx,6.6,True


## 6. Run extraction (with replay-from-cache)

For each attachment:

1. If a cached JSON exists in `reference/cu-output/<stem>.json` and `FORCE_REFRESH` is `False`, load it.
2. Otherwise, call `begin_analyze_binary` with the SOV file's bytes, poll the LRO, and save the raw result JSON to disk.

This gives us a deterministic offline replay for the workshop and an obvious cache-bust path for re-runs.


In [6]:
import zipfile
import shutil
import tempfile

CONTENT_TYPE_BY_SUFFIX = {
    ".pdf":  "application/pdf",
    ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    ".png":  "image/png",
    ".jpg":  "image/jpeg",
    ".jpeg": "image/jpeg",
}


def _result_to_dict(result) -> dict:
    """Convert an SDK AnalysisResult to a JSON-serializable dict."""
    if hasattr(result, "as_dict"):
        return result.as_dict()
    return json.loads(json.dumps(result, default=lambda o: getattr(o, "__dict__", str(o))))


def extract_embedded_images(xlsx_path: Path, out_dir: Path) -> list[Path]:
    """List images embedded in an .xlsx (under xl/media/), copy to out_dir, return paths."""
    out_dir.mkdir(parents=True, exist_ok=True)
    paths: list[Path] = []
    try:
        with zipfile.ZipFile(xlsx_path) as zf:
            media = [n for n in zf.namelist() if n.startswith("xl/media/")]
            for i, name in enumerate(sorted(media), start=1):
                ext = Path(name).suffix.lower() or ".png"
                target = out_dir / f"{xlsx_path.stem}_image{i:02d}{ext}"
                with zf.open(name) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                paths.append(target)
    except zipfile.BadZipFile:
        pass
    return paths


def _analyze_one(path: Path, analyzer_id: str, template: dict, content_type: str) -> tuple[dict, float]:
    """Single CU analyze call (binary input)."""
    client = get_client()
    ensure_analyzer(analyzer_id, template)
    print(f"  ⏳ {path.name}  ->  '{analyzer_id}'  ({content_type})")
    t0 = time.perf_counter()
    with open(path, "rb") as f:
        poller = client.begin_analyze_binary(
            analyzer_id=analyzer_id,
            binary_input=f.read(),
            content_type=content_type,
        )
    payload = _result_to_dict(poller.result())
    return payload, time.perf_counter() - t0


def _location_key(loc_obj: dict) -> tuple:
    """Stable key for matching location rows across xlsx + image fan-outs."""
    obj = loc_obj.get("valueObject", {}) if isinstance(loc_obj, dict) else {}

    def _val(field):
        f = obj.get(field) or {}
        for k in ("valueNumber", "valueInteger", "valueString"):
            if k in f and f[k] is not None:
                return f[k]
        return None

    num = _val("LocationNumber")
    street = (_val("Street") or "").strip().lower()
    return (num, street)


def _field_has_value(field: dict) -> bool:
    """True if a CU field dict carries an actual extracted value (not just type/confidence)."""
    if not isinstance(field, dict):
        return False
    for k in ("valueString", "valueNumber", "valueInteger", "valueDate", "valueBoolean", "valueArray", "valueObject"):
        if k in field and field[k] not in (None, "", [], {}):
            return True
    return False


def _merge_image_locations(primary_locs: list[dict], image_batches: list[list[dict]]) -> tuple[list[dict], int, int]:
    """Merge image-derived rows into the primary list.
    - Match on (LocationNumber, Street). For matches, fill any field on the primary row
      that has no extracted value with the image's value (image complements xlsx).
    - For new keys (image-only rows), append the image row tagged with _source.
    Returns (merged_list, num_added, num_field_complements)."""
    out = list(primary_locs)
    by_key = {_location_key(r): r for r in out}
    added = 0
    complements = 0
    for batch_idx, batch in enumerate(image_batches):
        for img_loc in batch:
            k = _location_key(img_loc)
            existing = by_key.get(k)
            img_obj = img_loc.get("valueObject", {}) if isinstance(img_loc, dict) else {}
            if existing is None:
                img_loc.setdefault("_source", f"image[{batch_idx}]")
                out.append(img_loc)
                by_key[k] = img_loc
                added += 1
                continue
            # Field-level complement: fill missing fields from image (verbatim, no clobber).
            ex_obj = existing.setdefault("valueObject", {})
            row_complemented = False
            for fname, ifield in img_obj.items():
                if not _field_has_value(ifield):
                    continue
                efield = ex_obj.get(fname)
                if efield is None or not _field_has_value(efield):
                    ex_obj[fname] = ifield
                    complements += 1
                    row_complemented = True
            if row_complemented:
                src = existing.get("_source", "xlsx")
                if isinstance(src, str) and "image" not in src:
                    existing["_source"] = f"{src}+image[{batch_idx}]"
    return out, added, complements


def analyze_file(path: Path) -> dict:
    """Run the appropriate approach for this file. Caches result JSON on disk.

    Approach 1: .pdf                            -> sovExtractV1   (1 call)
    Approach 2: .xlsx, no embedded images       -> sovGenerateV1  (1 call)
    Approach 3: .xlsx WITH embedded images      -> sovGenerateV1 on the
                                                   workbook + the same analyzer on
                                                   each embedded image; merge
                                                   Locations[] client-side (field-level
                                                   complement on matching rows, append
                                                   image-only rows).
    """
    cache_path = OUTPUT_DIR / f"{path.stem}.json"
    if cache_path.exists() and not FORCE_REFRESH:
        return json.loads(cache_path.read_text(encoding="utf-8"))

    suffix = path.suffix.lower()

    # Approach 3 detection: .xlsx with embedded images.
    if suffix == ".xlsx":
        with tempfile.TemporaryDirectory() as tmp:
            image_paths = extract_embedded_images(path, Path(tmp))
            if image_paths:
                print(f"  📎 found {len(image_paths)} embedded image(s) -- using Pattern C (workbook + image fan-out)")

                # 1. Workbook -> generate analyzer
                payload, elapsed_main = _analyze_one(
                    path,
                    ANALYZER_GENERATE_ID,
                    template_generate,
                    CONTENT_TYPE_BY_SUFFIX[".xlsx"],
                )

                # 2. Each embedded image -> the same generate analyzer (image content type)
                image_batches: list[list[dict]] = []
                elapsed_images = 0.0
                for img in image_paths:
                    img_payload, img_elapsed = _analyze_one(
                        img,
                        ANALYZER_GENERATE_ID,
                        template_generate,
                        CONTENT_TYPE_BY_SUFFIX[img.suffix.lower()],
                    )
                    elapsed_images += img_elapsed
                    img_locs = (
                        img_payload.get("contents", [{}])[0]
                                   .get("fields", {})
                                   .get("Locations", {})
                                   .get("valueArray", [])
                    )
                    image_batches.append(img_locs)

                # 3. Merge client-side (field-level complement + append new rows)
                primary_locs = (
                    payload.get("contents", [{}])[0]
                           .get("fields", {})
                           .get("Locations", {})
                           .get("valueArray", [])
                )
                merged, added, complements = _merge_image_locations(primary_locs, image_batches)
                try:
                    payload["contents"][0]["fields"]["Locations"]["valueArray"] = merged
                    # Pattern C also recomputes LocationCount: post-merge total = workbook + image-only rows.
                    lc_field = payload["contents"][0]["fields"].setdefault("LocationCount", {"type": "number"})
                    lc_field["type"] = "number"
                    lc_field["valueNumber"] = len(merged)
                except (KeyError, IndexError):
                    pass

                payload["_meta"] = {
                    "source_file":         path.name,
                    "approach":            "pattern-c-xlsx-plus-image-merge",
                    "analyzer_id":         ANALYZER_GENERATE_ID,
                    "image_calls":         len(image_paths),
                    "added_from_images":   added,
                    "field_complements":   complements,
                    "elapsed_main_sec":    round(elapsed_main, 2),
                    "elapsed_images_sec":  round(elapsed_images, 2),
                    "elapsed_total_sec":   round(elapsed_main + elapsed_images, 2),
                }
                cache_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
                print(f"  ✅ saved {cache_path.name}  (main {elapsed_main:.1f}s + images {elapsed_images:.1f}s, +{added} rows, +{complements} field complements)")
                return payload

    # Approaches 1 & 2: single binary input.
    if suffix == ".pdf":
        analyzer_id, template, approach = ANALYZER_EXTRACT_ID, template_extract, "standard-extract"
    elif suffix == ".xlsx":
        analyzer_id, template, approach = ANALYZER_GENERATE_ID, template_generate, "standard-generate"
    else:
        raise ValueError(f"Unsupported suffix: {suffix}")

    payload, elapsed = _analyze_one(path, analyzer_id, template, CONTENT_TYPE_BY_SUFFIX[suffix])
    payload["_meta"] = {
        "source_file":  path.name,
        "approach":     approach,
        "analyzer_id":  analyzer_id,
        "elapsed_sec":  round(elapsed, 2),
    }
    cache_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"  ✅ saved {cache_path.name}  ({elapsed:.1f}s)")
    return payload


results: dict[str, dict] = {}
for path in attachments:
    cache = (OUTPUT_DIR / f"{path.stem}.json").exists() and not FORCE_REFRESH
    print(f"[{ 'cache' if cache else 'live ' }] {path.name}")
    results[path.name] = analyze_file(path)

print(f"\nDone. {len(results)} file(s) processed.")


[live ] 01_acme_SOV.xlsx
  📎 found 1 embedded image(s) -- using Pattern C (workbook + image fan-out)
✅ Client ready  | endpoint=https://ais-aidemos-dev-01.services.ai.azure.com/  | auth=DefaultAzureCredential
♻️  Reusing analyzer 'sovGenerateV1'  (status: ContentAnalyzerStatus.READY)
  ⏳ 01_acme_SOV.xlsx  ->  'sovGenerateV1'  (application/vnd.openxmlformats-officedocument.spreadsheetml.sheet)
♻️  Reusing analyzer 'sovGenerateV1'  (status: ContentAnalyzerStatus.READY)
  ⏳ 01_acme_SOV_image01.png  ->  'sovGenerateV1'  (image/png)
  ✅ saved 01_acme_SOV.json  (main 19.7s + images 9.9s, +3 rows, +0 field complements)
[live ] 02_cascade_SOV.xlsx
♻️  Reusing analyzer 'sovGenerateV1'  (status: ContentAnalyzerStatus.READY)
  ⏳ 02_cascade_SOV.xlsx  ->  'sovGenerateV1'  (application/vnd.openxmlformats-officedocument.spreadsheetml.sheet)
  ✅ saved 02_cascade_SOV.json  (9.7s)
[live ] 03_magnolia_SOV.xlsx
♻️  Reusing analyzer 'sovGenerateV1'  (status: ContentAnalyzerStatus.READY)
  ⏳ 03_magnolia_SOV

## 7. Summarize the extracted fields

Walk each result's `fields` dict and pull out the headline values: insured name, effective date, broker, total insured value, and how many locations the analyzer recovered. The CU result shape returns each field as `{"type": ..., "valueX": ..., "confidence": ...}`, so we have a small helper to unwrap that.


In [7]:
def _unwrap(field: dict | None):
    """Return the scalar value from a CU field dict, regardless of type."""
    if not field:
        return None
    for key in ("valueString", "valueNumber", "valueDate",
                "valueInteger", "valueBoolean"):
        if key in field and field[key] is not None:
            return field[key]
    if "valueArray" in field:
        return field["valueArray"]
    if "valueObject" in field:
        return field["valueObject"]
    return None


def _confidence(field: dict | None) -> float | None:
    return field.get("confidence") if field else None


def summarize(payload: dict) -> dict:
    """Pull a one-row summary out of one Content Understanding result payload."""
    contents = (payload.get("result") or payload).get("contents", [])
    fields = contents[0].get("fields", {}) if contents else {}

    locations = _unwrap(fields.get("Locations")) or []
    return {
        "file":             payload.get("_meta", {}).get("source_file"),
        "insured":          _unwrap(fields.get("InsuredName")),
        "effective_date":   _unwrap(fields.get("EffectiveDate")),
        "broker":           _unwrap(fields.get("BrokerName")),
        "tiv":              _unwrap(fields.get("TotalInsuredValue")),
        "loc_count_field":  _unwrap(fields.get("LocationCount")),
        "loc_count_actual": len(locations),
        "conf_insured":     _confidence(fields.get("InsuredName")),
        "conf_tiv":         _confidence(fields.get("TotalInsuredValue")),
    }


summary_df = pd.DataFrame([summarize(r) for r in results.values()])
summary_df


,file,insured,effective_date,broker,tiv,loc_count_field,loc_count_actual,conf_insured,conf_tiv
0,01_acme_SOV.xlsx,"Acme Manufacturing & Distribution, Inc.",2026-07-01,Sterling Risk Partners,370100000.0,22,22,1.000,1.000
1,02_cascade_SOV.xlsx,NaN,NaN,NaN,NaN,8,8,NaN,NaN
2,03_magnolia_SOV.xlsx,Magnolia Hospitality Group LLC,2026-09-01,Crescent Insurance Services,510500000.0,15,15,1.000,1.000
3,04_summit_SOV.pdf,Summit Outdoor Retail Co.,2026-10-01,Rocky Mountain Brokerage,105200000.0,12,12,0.651,0.588
4,05_heartland_SOV.pdf,Heartland Agri-Processors Inc.,2026-11-15,Prairie State Insurance Agency,302000000.0,10,10,0.665,0.609
5,06_coastal_SOV.xlsx,Coastal Marine Services Inc.,2026-12-01,ATLANTIC SPECIALTY BROKERS,29600000.0,6,6,1.000,1.000


## 9. Cleanup (optional)

Delete the analyzer if you're done. Cached results in `cu-output/` are kept either way so the notebook stays replayable.


In [8]:
# Uncomment to delete the analyzer when you're done iterating.
# get_client().delete_analyzer(analyzer_id=ANALYZER_ID)
# print(f"🗑️  Deleted analyzer '{ANALYZER_ID}'.")
